# CDF Playground

Interactive exploration of empirical-CDF figures (`sinr_cdf`, `scnr_cdf`).

**Workflow.**  Run a CDF experiment first — e.g. `make sinr_cdf` — to
populate `results/exp_sinr_cdf/<ts>/`.  Set `exp_name` below to the
metric family (`'sinr'` or `'scnr'`) and iterate on the cells.
Switching between SINR and SCNR is one edit; all subsequent cells
derive their experiment folder, metric names, and titles from
`exp_name`.

**Compatible experiments:** `sinr_cdf`, `scnr_cdf`
(both use `kind == 'single'` and the `plot_cdf` function).

In [1]:
# Stage 12: shared setup — make `cordis` importable when this notebook
# is launched from notebooks/, then apply the IEEE paper rcParams.
import sys, logging
from pathlib import Path
from _playground_helpers import (
    setup_paper_style, load_latest_result, load_run, summarize,
)

import numpy as np
import matplotlib.pyplot as plt

# Set use_latex=False if pdflatex isn't on PATH (e.g. on a compute node).
setup_paper_style(use_latex=True)

logging.basicConfig(level=logging.WARNING, format='%(levelname)-7s %(message)s')

In [2]:
# ── The one knob: which metric family to load ──
exp_name = 'sinr'              # 'sinr' or 'scnr'

# Optional: load a specific run directory instead of the latest.
# Set to a path like 'results/exp_sinr_cdf/20260520_113500'
# (relative to the repo root) to re-render an older comparison.
EXP_DIR = None

# Derived names — change `exp_name` above and these update automatically.
EXPERIMENT     = f'{exp_name}_cdf'        # 'sinr_cdf' / 'scnr_cdf'
METRIC_MIN     = f'min_{exp_name}_db'     # 'min_sinr_db' / 'min_scnr_db'
METRIC_MEAN    = f'mean_{exp_name}_db'    # 'mean_sinr_db' / 'mean_scnr_db'
METRIC_DISPLAY = exp_name.upper()         # 'SINR' / 'SCNR'  (for titles)

result = load_result(EXPERIMENT, EXP_DIR)
assert result.kind == 'single', (
    f'This notebook is for CDF (kind="single") experiments; '
    f'got kind={result.kind}.  Use playground_sweep / _trace / _table.'
)
summarize(result)

NameError: name 'load_result' is not defined

## 1. Quick CDF — all algorithms

In [ ]:
from cordis.plotting import plot_cdf, figsize

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_cdf(result.sim_result, metric=METRIC_MIN, ax=ax)
ax.set_title(f'{EXPERIMENT}: min-{METRIC_DISPLAY} CDF')
ax.set_xlabel(f'min-{METRIC_DISPLAY} [dB]')
plt.show()

## 2. Filter algorithms via `only=`

`plot_cdf` accepts an `only=` list of algorithm display names — anything
not in the list is silently skipped.  Colors/markers stay consistent with
`ALGORITHM_STYLE` even when you drop algorithms.

In [ ]:
# Pick just the algorithms you want — easy A/B comparison.
PICK = ['CORDIS-Split', 'CORDIS-ADMM', 'Centralized']

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_cdf(result.sim_result, metric=METRIC_MIN, ax=ax, only=PICK)
ax.set_title(f'{EXPERIMENT}: CORDIS vs. Centralized')
ax.set_xlabel(f'min-{METRIC_DISPLAY} [dB]')
plt.show()

## 3. Alternate metrics — min vs. mean

The SimResult typically carries several metrics per run.  For the
currently-selected family (SINR or SCNR), the canonical pair is
`min_<family>_db` and `mean_<family>_db`.  Comparing the two side-
by-side shows how algorithm rankings shift between the worst-user
view and the average view.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=figsize(width='double', aspect=3/2))
plot_cdf(result.sim_result, metric=METRIC_MIN, ax=axes[0], only=PICK)
axes[0].set_title(f'Worst-user {METRIC_DISPLAY}')
axes[0].set_xlabel(f'min-{METRIC_DISPLAY} [dB]')
plot_cdf(result.sim_result, metric=METRIC_MEAN, ax=axes[1], only=PICK)
axes[1].set_title(f'Average-user {METRIC_DISPLAY}')
axes[1].set_xlabel(f'mean-{METRIC_DISPLAY} [dB]')
plt.tight_layout()
plt.show()

## 4. Per-algorithm style override

`ALGORITHM_STYLE` is a dict; monkey-patch entries for one-off tweaks
without committing changes to `cordis/plotting/style.py`.

In [ ]:
from cordis.plotting.style import ALGORITHM_STYLE
import copy

# Snapshot original so cell is re-runnable.
_orig = copy.deepcopy(ALGORITHM_STYLE)

# Bump CORDIS-Split to a heavier line, switch its color.
if 'CORDIS-Split' in ALGORITHM_STYLE:
    ALGORITHM_STYLE['CORDIS-Split']['linewidth'] = 2.5
    ALGORITHM_STYLE['CORDIS-Split']['color']     = '#d62728'

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_cdf(result.sim_result, metric=METRIC_MIN, ax=ax, only=PICK)
ax.set_title('Style override demo (red, thicker Split)')
plt.show()

# Restore the original style so subsequent cells aren't affected.
ALGORITHM_STYLE.clear(); ALGORITHM_STYLE.update(_orig)

## 5. Tail behavior — log x-axis

Useful when the interesting story is in the outage region
(e.g. 5th-percentile SINR).

In [ ]:
fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
plot_cdf(result.sim_result, metric=METRIC_MIN, ax=ax, only=PICK)
# Mark the 5%/50%/95% lines
for q in (0.05, 0.5, 0.95):
    ax.axhline(q, color='lightgray', linestyle=':', linewidth=0.8, zorder=0)
ax.set_ylim(0, 1)
plt.show()

## 6. Figsize variants

In [ ]:
# Figsize variants — `figsize` returns (w, h) in inches for matplotlib.
# width: 'single' (one column), 'double' (two-column), 'third' (3-up panel).
# aspect: w/h ratio.  Tweak both to fit your paper layout.
from cordis.plotting import figsize

for width in ('single', 'double', 'third'):
    w, h = figsize(width=width, aspect=3/2)
    print(f'{width:>6}: ({w:.2f}, {h:.2f}) inches')

# Example: tight three-up panel for a paper sub-figure
# fig, axes = plt.subplots(1, 3, figsize=figsize(width='double', aspect=3.5/1.5))

## 7. Save with provenance metadata

In [ ]:
# Save with provenance metadata (Git SHA, creation date, etc. — embedded
# into the PDF's metadata, prepended as comments in the .pgf).
from cordis.plotting import save_figure

# Adjust EXPERIMENT and metric labels to match the figure above.
out = save_figure(
    fig,
    base_path=f'../figures/playground/{EXPERIMENT}_demo',
    formats=('pdf', 'png'),       # add 'pgf' on systems with LaTeX
    metadata={'Experiment': EXPERIMENT, 'Notebook': 'playground'},
)
for p in out:
    print('wrote', p)

## 8. Multi-run overlay

Compare two runs of the same experiment — e.g. before vs. after
tuning a parameter — by loading both and overlaying their curves.

Set `RUN_A` / `RUN_B` to two specific run paths to enable this cell;
remove them or set to `None` to skip.

In [ ]:
RUN_A = None    # e.g. 'results/exp_sinr_cdf/2026-05-18_12-00-00'
RUN_B = None    # e.g. 'results/exp_sinr_cdf/2026-05-19_15-30-00'

if RUN_A and RUN_B:
    res_a = load_run(RUN_A)
    res_b = load_run(RUN_B)
    fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
    # plot_cdf doesn't accept linestyle/label_suffix kwargs, so we
    # post-process the Line2D objects to differentiate the two runs.
    n_before_a = len(ax.lines)
    plot_cdf(res_a.sim_result, metric=METRIC_MIN, ax=ax, only=PICK)
    n_before_b = len(ax.lines)
    for ln in ax.lines[n_before_a:n_before_b]:
        ln.set_linestyle('--')
        ln.set_label(ln.get_label() + ' (A)')
    plot_cdf(res_b.sim_result, metric=METRIC_MIN, ax=ax, only=PICK)
    for ln in ax.lines[n_before_b:]:
        ln.set_label(ln.get_label() + ' (B)')
    ax.legend(loc='best')        # refresh legend after relabeling
    plt.show()
else:
    print('Set RUN_A and RUN_B above to compare two runs.')